# Fine-tuning Llama-3.2-3B with vanilla Hugging Face (LoRA SFT)

This notebook reproduces the **exact same experiment** as
`01_unsloth_lora_sft_clean.ipynb` — same model, dataset, LoRA settings, SFT
hyperparameters, response-only objective, and inference test — but using the
standard Hugging Face stack (`transformers`, `datasets`, `peft`, `trl`,
`bitsandbytes`) instead of Unsloth.

The goal is to **see what Unsloth abstracts away**. Each section notes the
Unsloth one-liner and the vanilla code it corresponds to.

> Steps 2 and 3 of the reference outline (load model / configure 4-bit) are
> combined below, because the quantization config must be passed *at load time*.


## 1. Setup

We pin the same `transformers` / `trl` versions used by the Unsloth notebook so
the two are directly comparable and the APIs below match.

In [ ]:
%%capture
!pip install "transformers==4.56.2" "trl==0.22.2" "peft>=0.13.0" \
             "datasets==4.3.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" \
             sentencepiece protobuf

## 2. Load Model and Tokenizer (with 4-bit quantization)

| Unsloth | Hugging Face |
|---|---|
| `FastLanguageModel.from_pretrained(..., load_in_4bit=True)` | `BitsAndBytesConfig` + `AutoModelForCausalLM.from_pretrained` + `AutoTokenizer` |

Unsloth's single `load_in_4bit=True` hides a `BitsAndBytesConfig`, the compute
dtype choice, device placement, and padding-token setup. Here we do each
explicitly.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

max_seq_length = 2048
model_name = "unsloth/Llama-3.2-3B-Instruct"   # same weights as NB1
# meta-llama/Llama-3.2-3B-Instruct is the gated original (needs an HF token)

# bf16 on Ampere+ (RTX 30xx / A100), fp16 on older cards (Tesla T4)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# This is what Unsloth's load_in_4bit=True configures under the hood:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",              # NormalFloat4
    bnb_4bit_compute_dtype = compute_dtype,   # matmuls run in this dtype
    bnb_4bit_use_double_quant = True,         # nested quant of the quant constants
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    dtype = compute_dtype,        # 'torch_dtype' was renamed to 'dtype' in transformers 4.56
    device_map = {"": 0},         # whole model on GPU 0
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Unsloth sets a padding token for us; vanilla HF usually needs it done manually.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

## 3–4. Prepare for k-bit Training and Configure LoRA

| Unsloth | Hugging Face |
|---|---|
| `FastLanguageModel.get_peft_model(..., use_gradient_checkpointing="unsloth")` | `prepare_model_for_kbit_training` + `LoraConfig` + `get_peft_model` |

`prepare_model_for_kbit_training` casts layernorms to fp32, enables input-grads,
and turns on gradient checkpointing — the housekeeping Unsloth does silently. The
`LoraConfig` fields map one-to-one to Unsloth's `get_peft_model` arguments.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ≈ Unsloth's use_gradient_checkpointing="unsloth"
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False   # incompatible with gradient checkpointing while training

lora_config = LoraConfig(
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Load Dataset

Identical to the Unsloth notebook.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("mlabonne/FineTome-100k", split = "train")
dataset[5]["conversations"]

## 6. Standardize Conversation Format

| Unsloth | Hugging Face |
|---|---|
| `standardize_sharegpt(dataset)` | manual `map` from `{"from","value"}` to `{"role","content"}` |

ShareGPT uses `from`/`value` with roles `human`/`gpt`/`system`. We remap them to
the HF chat schema by hand.

In [ ]:
role_map = {"human": "user", "gpt": "assistant", "system": "system",
            "user": "user", "assistant": "assistant"}

def standardize(example):
    convo = [{"role": role_map[turn["from"]], "content": turn["value"]}
             for turn in example["conversations"]]
    return {"messages": convo}

dataset = dataset.map(standardize, remove_columns=dataset.column_names)
dataset[5]["messages"]

## 7–8. Apply Chat Template and Build Training Text

| Unsloth | Hugging Face |
|---|---|
| `get_chat_template(tokenizer, "llama-3.1")` | use the tokenizer's **built-in** chat template |

The instruct tokenizer already ships with the Llama-3 chat template, so
`apply_chat_template` works without installing one. We render each conversation
to a single `text` string, exactly as the Unsloth notebook does.

In [ ]:
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize = False, add_generation_prompt = False)
    return {"text": text}

dataset = dataset.map(formatting_func)
print(dataset[5]["text"][:1000])

## 9. Tokenize and Apply Response-Only Loss Masking

| Unsloth | Hugging Face |
|---|---|
| `train_on_responses_only(trainer)` | manual `-100` label masking of non-assistant tokens |

This is the step Unsloth most conveniently hides. Modern TRL offers
`assistant_only_loss=True`, but it requires the chat template to contain
`{% generation %}` / `{% endgeneration %}` markers so the tokenizer can emit an
assistant-token mask — and the stock Llama-3.2 template has none. So we replicate
`train_on_responses_only` directly: find each assistant header, keep the tokens
of the response that follows (up to the next user turn), and set every other
label to `-100`.

We tokenize with `add_special_tokens=False` because the chat template already
rendered the leading `<|begin_of_text|>`.

In [ ]:
response_template    = "<|start_header_id|>assistant<|end_header_id|>\n\n"
instruction_template = "<|start_header_id|>user<|end_header_id|>\n\n"

resp_ids  = tokenizer.encode(response_template, add_special_tokens=False)
instr_ids = tokenizer.encode(instruction_template, add_special_tokens=False)

def find_sublist_positions(seq, sub):
    m = len(sub)
    return [i for i in range(len(seq) - m + 1) if seq[i:i+m] == sub]

def tokenize_and_mask(example):
    input_ids = tokenizer(example["text"], add_special_tokens=False,
                          truncation=True, max_length=max_seq_length)["input_ids"]
    labels = [-100] * len(input_ids)
    resp_pos  = find_sublist_positions(input_ids, resp_ids)
    instr_pos = find_sublist_positions(input_ids, instr_ids)
    for rp in resp_pos:
        start = rp + len(resp_ids)                       # first token of the response
        nxt = min([ip for ip in instr_pos if ip > rp],   # next user turn (or end)
                  default=len(input_ids))
        for j in range(start, nxt):
            labels[j] = input_ids[j]
    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}

tokenized = dataset.map(tokenize_and_mask, remove_columns=dataset.column_names)

## 10. Inspect Tokenized Inputs and Labels

The same check as the Unsloth notebook. First the full input:

In [ ]:
tokenizer.decode(tokenized[5]["input_ids"])

Now the labels, with masked positions shown as spaces — only the assistant response should survive:

In [ ]:
space = tokenizer(" ", add_special_tokens=False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in tokenized[5]["labels"]])

## 11. Configure SFT Training

| Unsloth | Hugging Face |
|---|---|
| `SFTTrainer(...)` (patched) | TRL `SFTConfig` + `SFTTrainer` |

Two API notes for the pinned versions: `max_seq_length` became `max_length` in
`SFTConfig` (TRL 0.12+), and the `tokenizer` argument became `processing_class`.
Because our dataset is **already tokenized and masked**, we tell the trainer to
skip its own preprocessing with `skip_prepare_dataset`, and pad with
`DataCollatorForSeq2Seq` (the same collator the Unsloth notebook uses).

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

sft_config = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
    max_length = max_seq_length,     # renamed from max_seq_length in TRL 0.12+
    packing = False,
    dataset_kwargs = {"skip_prepare_dataset": True},  # data is already tokenized
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,    # 'tokenizer' was renamed to 'processing_class'
    train_dataset = tokenized,
    args = sft_config,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
)

## 12. Inspect GPU Memory

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

## 13. Train

In [ ]:
trainer_stats = trainer.train()

## 14. Training / Memory Statistics

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 15. Switch to Inference

| Unsloth | Hugging Face |
|---|---|
| `FastLanguageModel.for_inference(model)` | `model.eval()` + re-enable `use_cache` |

In [ ]:
model.config.use_cache = True   # was disabled for gradient checkpointing
model.eval()

## 16. Run the Fibonacci Inference Test

Same prompt and sampling settings as the Unsloth notebook. One difference:
vanilla `generate` needs `do_sample=True` for `temperature` / `min_p` to take
effect (Unsloth turns sampling on implicitly). Outputs won't match the Unsloth
run token-for-token because sampling is stochastic across the two stacks.

In [ ]:
messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,   # required for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs, max_new_tokens = 64, use_cache = True,
    do_sample = True, temperature = 1.5, min_p = 0.1,
    pad_token_id = tokenizer.pad_token_id,
)

## 17. Decode the Output

In [ ]:
tokenizer.batch_decode(outputs)

## Summary: what Unsloth abstracts

| Stage | Unsloth | Vanilla Hugging Face |
|---|---|---|
| Load + quantize | `from_pretrained(load_in_4bit=True)` | `BitsAndBytesConfig` + `AutoModelForCausalLM` + manual dtype/device/pad token |
| LoRA | `get_peft_model(use_gradient_checkpointing="unsloth")` | `prepare_model_for_kbit_training` + `LoraConfig` + `get_peft_model` |
| ShareGPT format | `standardize_sharegpt` | manual role remap |
| Chat template | `get_chat_template(..., "llama-3.1")` | tokenizer's built-in template |
| Response-only loss | `train_on_responses_only(trainer)` | manual `-100` label masking |
| Training | patched `SFTTrainer` | TRL `SFTConfig` + `SFTTrainer` (`max_length`, `processing_class`) |
| Inference mode | `for_inference(model)` | `model.eval()` + `use_cache=True` |

Unsloth's value is speed and fewer moving parts; the vanilla stack is the same
pipeline with every step made explicit — plus its custom fused kernels for the
throughput and memory wins.